# Envenenamento de dados e modelos: o grupo que ensinou a aceitar atraso

OWASP Top 10 para LLMs 2025 — **LLM04 Data & Model Poisoning**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
O Aurora Assist é ajustado continuamente a partir de avaliações "útil"/"não útil" das suas respostas. Um grupo coordenado vota em massa para ensinar que prazos vencidos "ainda são aceitos".

In [2]:
RESPOSTA_CORRETA = "Prazos vencidos não são aceitos sem justificativa protocolada."
RESPOSTA_ENVENENADA = "Prazos vencidos ainda são aceitos, basta avisar depois."

def simular_votacao(rodadas, votos_por_rodada, grupo_tamanho, com_defesa):
    """O grupo coordenado tem só `grupo_tamanho` identidades distintas, mas revota a cada
    rodada tentando votar `votos_por_rodada` vezes. Cada voto 'útil' soma 1 ao peso da
    resposta envenenada. Com defesa, uma mesma identidade só conta UMA vez em toda a
    simulação (limite de influência por fonte/proveniência)."""
    peso = 1  # peso inicial, igual ao da resposta correta
    historico = [peso]
    ja_contadas = set()
    for _ in range(rodadas):
        incremento = 0
        for i in range(votos_por_rodada):
            ident = "coordenado-{}".format(i % grupo_tamanho)
            if com_defesa:
                if ident in ja_contadas:
                    continue
                ja_contadas.add(ident)
            incremento += 1
        peso += incremento
        historico.append(peso)
    return historico


## Sem defesa
Todo voto conta, mesmo vindo repetidamente das mesmas poucas contas do grupo coordenado (sem checar proveniência).

In [3]:
historico = simular_votacao(rodadas=5, votos_por_rodada=20, grupo_tamanho=5, com_defesa=False)
print("peso da resposta envenenada por rodada:", historico)
prob_envenenada = historico[-1] / (historico[-1] + 1)
print("probabilidade de sortear a resposta envenenada ao final:", round(prob_envenenada, 2))
assert prob_envenenada > 0.9


peso da resposta envenenada por rodada: [1, 21, 41, 61, 81, 101]
probabilidade de sortear a resposta envenenada ao final: 0.99


## Com defesa
Cada identidade só pode contar um voto em toda a simulação (checagem simples de proveniência/limite de influência por fonte).

In [4]:
historico_d = simular_votacao(rodadas=5, votos_por_rodada=20, grupo_tamanho=5, com_defesa=True)
print("peso da resposta envenenada por rodada (com defesa):", historico_d)
prob_envenenada_d = historico_d[-1] / (historico_d[-1] + 1)
print("probabilidade de sortear a resposta envenenada ao final:", round(prob_envenenada_d, 2))
assert prob_envenenada_d < prob_envenenada  # reduz o efeito, mas o grupo ainda tem alguma influência


peso da resposta envenenada por rodada (com defesa): [1, 6, 6, 6, 6, 6]
probabilidade de sortear a resposta envenenada ao final: 0.86


## O que observar
- O efeito é cumulativo e silencioso: nenhuma rodada isolada parece um ataque.
- "Identidades únicas" aqui é uma simplificação didática de um problema real e difícil (detecção de voto coordenado/sybil).
- A defesa reduz a velocidade do envenenamento, não a elimina sozinha — por isso este risco também pede amostragem humana e testes adversariais antes de publicar uma nova versão.